# **Breast Cancer Classification with a simple Neural Network (NN)**

Previous videos:
*   Brest Cancer Classification with Logistic Regression video: https://youtu.be/bFh1umUDaGc
*   EDA video: https://youtu.be/imdS1LIlISY
*   Data Visualization video: https://youtu.be/gFSerq21wo8

---

This notebook presents an **industry-oriented, production-ready machine learning system** designed to classify breast tumor biopsies as **Malignant (0)** or **Benign (1)** using the Wisconsin Breast Cancer Dataset.

### Project Highlights:
1. **Exploratory Data Analysis (EDA)** and 2D Principal Component Analysis (PCA) dataset visualization.
2. **Hyperparameter Tuning** via **Optuna** to optimize Logistic Regression, Random Forest, XGBoost, and Scikit-Learn MLP (Neural Network) classifiers.
3. **Clinical Metric Alignment** by optimizing and ranking models based on **Malignant Recall (Sensitivity)** to ensure critical cases are not missed.
4. **Global and Local Model Interpretability** using **SHAP (SHapley Additive exPlanations)** to understand the clinical feature contribution.
5. **Production Model Export** exporting the standard scaler and the best performing tuned estimator.

In [ ]:
# Importing dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
print("Setup completed successfully.")

## 1. Data Collection & Preprocessing

We load the Wisconsin Breast Cancer dataset, split it into **Train (70%)**, **Validation (10%)**, and **Test (20%)** sets, and normalize all features utilizing the `StandardScaler` fitted only on the training set to prevent data leakage.

In [ ]:
# Load the dataset
cancer = sklearn.datasets.load_breast_cancer()
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['label'] = cancer.target

print(f"Dataset dimensions: {df.shape}")
print(f"Class distribution: Benign (1) = {sum(df['label'] == 1)}, Malignant (0) = {sum(df['label'] == 0)}")
df.head()

In [ ]:
# Separate features & label
X = df.drop(columns='label')
Y = df['label']

# Split into Train+Val (80%) and Test (20%)
X_train_val, X_test, Y_train_val, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

# Split Train+Val into Train (70%) and Val (10%)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train_val, Y_train_val, test_size=0.125, random_state=42, stratify=Y_train_val
)

# Standardize features
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_val_std = scaler.transform(X_val)
X_test_std = scaler.transform(X_test)

print(f"Train size: {X_train_std.shape[0]}, Val size: {X_val_std.shape[0]}, Test size: {X_test_std.shape[0]}")

## 2. Exploratory Data Visualization & PCA

We reduce the 30 features into 2 Principal Components (PC) to visually inspect feature separability.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pcs = pca.fit_transform(X_train_std)
pca_df = pd.DataFrame(pcs, columns=['PC1', 'PC2'])
pca_df['Diagnosis'] = Y_train.map({0: 'Malignant', 1: 'Benign'}).values

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Diagnosis', palette={'Malignant': '#d11a5e', 'Benign': '#2d6a4f'}, alpha=0.8)
plt.title(f"2D PCA Projection (Explained Variance: {sum(pca.explained_variance_ratio_):.2%})")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

## 3. Optuna Hyperparameter Tuning

We run Optuna tuning loops to find the best parameters, optimizing for **0.5 * Recall (Malignant class) + 0.5 * ROC-AUC**.

In [ ]:
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import recall_score, roc_auc_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Tuning utilities ready.")

In [ ]:
# Tuning Logistic Regression
def lr_objective(trial):
    C = trial.suggest_float('C', 1e-4, 1e2, log=True)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
    solver = 'liblinear' if penalty == 'l1' else 'lbfgs'
    
    clf = LogisticRegression(C=C, penalty=penalty, solver=solver, random_state=42, max_iter=1000)
    clf.fit(X_train_std, Y_train)
    
    preds = clf.predict(X_val_std)
    probs = clf.predict_proba(X_val_std)[:, 1]
    
    # Metric focused on cancer detection (Recall on class 0)
    recall = recall_score(Y_val, preds, pos_label=0)
    roc_auc = roc_auc_score(Y_val, probs)
    return 0.5 * recall + 0.5 * roc_auc

study_lr = optuna.create_study(direction='maximize')
study_lr.optimize(lr_objective, n_trials=20)
print(f"Best LR params: {study_lr.best_params}")

solver = 'liblinear' if study_lr.best_params['penalty'] == 'l1' else 'lbfgs'
lr_best = LogisticRegression(C=study_lr.best_params['C'], penalty=study_lr.best_params['penalty'], solver=solver, random_state=42)
lr_best.fit(X_train_std, Y_train)

In [ ]:
# Tuning XGBoost
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    
    clf = xgb.XGBClassifier(**params)
    clf.fit(X_train_std, Y_train)
    
    preds = clf.predict(X_val_std)
    probs = clf.predict_proba(X_val_std)[:, 1]
    
    recall = recall_score(Y_val, preds, pos_label=0)
    roc_auc = roc_auc_score(Y_val, probs)
    return 0.5 * recall + 0.5 * roc_auc

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(xgb_objective, n_trials=20)
print(f"Best XGBoost params: {study_xgb.best_params}")

xgb_best = xgb.XGBClassifier(**study_xgb.best_params, random_state=42)
xgb_best.fit(X_train_std, Y_train)

In [ ]:
# Tuning MLP Classifier Neural Network
def mlp_objective(trial):
    layer1 = trial.suggest_categorical('layer1', [16, 32, 64])
    layer2 = trial.suggest_categorical('layer2', [8, 16, 32])
    alpha = trial.suggest_float('alpha', 1e-5, 1e-1, log=True)
    learning_rate = trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True)
    
    clf = MLPClassifier(hidden_layer_sizes=(layer1, layer2), alpha=alpha, learning_rate_init=learning_rate, max_iter=300, early_stopping=True, random_state=42)
    clf.fit(X_train_std, Y_train)
    
    preds = clf.predict(X_val_std)
    probs = clf.predict_proba(X_val_std)[:, 1]
    
    recall = recall_score(Y_val, preds, pos_label=0)
    roc_auc = roc_auc_score(Y_val, probs)
    return 0.5 * recall + 0.5 * roc_auc

study_mlp = optuna.create_study(direction='maximize')
study_mlp.optimize(mlp_objective, n_trials=15)
print(f"Best MLP Neural Network params: {study_mlp.best_params}")

mlp_best = MLPClassifier(hidden_layer_sizes=(study_mlp.best_params['layer1'], study_mlp.best_params['layer2']), alpha=study_mlp.best_params['alpha'], learning_rate_init=study_mlp.best_params['learning_rate_init'], max_iter=400, early_stopping=True, random_state=42)
mlp_best.fit(X_train_std, Y_train)

## 4. Evaluation and Leaderboard

We evaluate all tuned models on the test set, computing accuracy, precision, recall, f1, and roc_auc. Metrics are computed specifically for class 0 (Malignant).

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix, roc_curve

models = {
    "Logistic Regression": lr_best,
    "XGBoost": xgb_best,
    "Neural Network (MLP)": mlp_best
}

leaderboard = []
for name, clf in models.items():
    preds = clf.predict(X_test_std)
    probs = clf.predict_proba(X_test_std)[:, 1]
    
    # Metrics for Malignant (class 0)
    acc = accuracy_score(Y_test, preds)
    prec = precision_score(Y_test, preds, pos_label=0)
    rec = recall_score(Y_test, preds, pos_label=0)
    f1 = f1_score(Y_test, preds, pos_label=0)
    auc_val = roc_auc_score(Y_test, probs)
    
    leaderboard.append({
        'Model': name,
        'Accuracy': acc,
        'Precision (Malignant)': prec,
        'Recall (Malignant)': rec,
        'F1-Score (Malignant)': f1,
        'ROC-AUC': auc_val
    })
    
leaderboard_df = pd.DataFrame(leaderboard).sort_values(by='Recall (Malignant)', ascending=False)
leaderboard_df

In [ ]:
# Plotting Confusion Matrix for the best model
best_model_name = leaderboard_df.iloc[0]['Model']
best_clf = models[best_model_name]
best_preds = best_clf.predict(X_test_std)
cm = confusion_matrix(Y_test, best_preds)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Malignant (0)', 'Benign (1)'],
            yticklabels=['Malignant (0)', 'Benign (1)'])
plt.title(f"Confusion Matrix - {best_model_name}")
plt.ylabel("Actual Label")
plt.xlabel("Predicted Label")
plt.show()

## 5. Model Interpretability using SHAP

We explain the top tree model (XGBoost) using SHAP values. Features that have high values pointing left push predictions towards the **Malignant** class, while features pushing right favor the **Benign** class.

In [ ]:
import shap

explainer = shap.TreeExplainer(xgb_best)
shap_values = explainer.shap_values(X_train_std)

# For binary classification target, class 0 (Malignant) is 1.0 - Benign probability.
# We explain predictions relative to the Malignant class.
if len(shap_values.shape) == 3: # multi-output check
    shap_values = shap_values[:, :, 0]

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_train_std, feature_names=cancer.feature_names, show=False)
plt.title("SHAP Feature Influences - Malignant Class (XGBoost)", fontsize=14, pad=15)
plt.show()